# 05 — Train + Eval + Forensic Interpretability (single H100)

**Goal:** push toward an ARC-AGI top-100 result on a single H100 with the full modern alignment stack — **base → SFT → DPO → GRPO** — and capture *deep probe traces* at every stage so we can render 11 interactive Plotly views explaining what training actually did to the model.

**Hardware:** Colab Pro+ with H100 (80GB). Falls back to A100/L4 with reduced batch.

**Why this notebook exists alongside `02_train_h100`:**
- `02_train_h100` does SFT only; this one adds DPO (preference) + GRPO (RL with verifiable reward) — the same recipe top public ARC fine-tunes use.
- Captures hidden states, layer norms, top neurons, LoRA deltas, gradient flows at every stage so you can **see** how training rewires the network.
- Outputs 11 standalone HTML viz files in `runs/<RUN_ID>/viz/` — open them in any browser without re-running.

**Optimization stack:** QLoRA 4-bit + LoRA r=64 on attn+MLP + FlashAttention-2 + bf16 + paged_adamw_8bit + gradient checkpointing + D4×color-perm augmentation.


In [ ]:
# 0.1 — Hardware check
import torch, sys
assert torch.cuda.is_available(), "Need a CUDA GPU. Switch Colab runtime to GPU (H100/A100/L4)."
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {GPU_NAME} ({VRAM_GB:.0f} GB) | torch {torch.__version__} | python {sys.version.split()[0]}")
IS_H100 = "H100" in GPU_NAME


In [ ]:
# 0.2 — Install dependencies (latest known-good for Qwen3 + GRPO)
# Qwen3 needs transformers>=4.51, GRPOTrainer needs trl>=0.15.
!pip -q install -U \
    "transformers>=4.51,<4.55" \
    "trl>=0.15,<0.20" \
    "peft>=0.14" \
    "accelerate>=1.4" \
    "bitsandbytes>=0.45" \
    "datasets>=3.2" \
    "plotly>=5.24" \
    sentencepiece huggingface_hub


In [ ]:
# 0.3 — Run config + paths
import os, time, json, random
from pathlib import Path

# --- pick model ---
# Qwen3-8B is the natural top-100 starting point on a single H100 with the full 4-stage stack.
# Bump to 14B if you have headroom; drop to 4B for fast iteration.
MODEL_ID = "Qwen/Qwen3-8B"

RUN_ID   = f"run-{int(time.time())}"
ROOT     = Path("/content") if Path("/content").exists() else Path(".")
RUN_DIR  = ROOT / "runs" / RUN_ID
for sub in ["adapters/sft", "adapters/dpo", "adapters/grpo",
            "traces/base", "traces/sft", "traces/dpo", "traces/grpo",
            "logs", "viz", "eval"]:
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)

DATA_DIR = ROOT / "arc_data"; DATA_DIR.mkdir(exist_ok=True)
SEED = 0
random.seed(SEED); torch.manual_seed(SEED)

print("RUN_DIR:", RUN_DIR)
print("MODEL  :", MODEL_ID)


## 1 — Data: ARC-AGI + RE-ARC + BARC, with D4×color-perm augmentation

Same recipe as `02_train_h100`. The single biggest data trick is multiplying every base task by the 8-element dihedral group × random color permutations.


In [ ]:
# 1.1 — Download ARC-AGI-1
import urllib.request, zipfile, io
if not (DATA_DIR / "training").exists():
    url = "https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip"
    print("Downloading ARC-AGI...")
    with urllib.request.urlopen(url) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(DATA_DIR)
    src = DATA_DIR / "ARC-AGI-master" / "data"
    for kind in ["training", "evaluation"]:
        (DATA_DIR / kind).mkdir(exist_ok=True)
        for f in (src / kind).glob("*.json"):
            (DATA_DIR / kind / f.name).write_bytes(f.read_bytes())

arc_train = [json.loads(p.read_text()) for p in sorted((DATA_DIR/"training").glob("*.json"))]
arc_eval  = [(p.stem, json.loads(p.read_text())) for p in sorted((DATA_DIR/"evaluation").glob("*.json"))]
print(f"ARC: {len(arc_train)} train, {len(arc_eval)} eval")


In [ ]:
# 1.2 — RE-ARC (procedural re-generations of ARC tasks)
re_arc_dir = DATA_DIR / "re_arc"
re_arc_tasks = []
if not re_arc_dir.exists():
    try:
        with urllib.request.urlopen("https://github.com/michaelhodel/re-arc/raw/main/re_arc.zip") as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(re_arc_dir)
    except Exception as e:
        print("RE-ARC skipped:", e)
if re_arc_dir.exists():
    for fp in sorted(re_arc_dir.rglob("*.json"))[:5000]:
        try: re_arc_tasks.append(json.loads(fp.read_text()))
        except: pass
print(f"RE-ARC tasks: {len(re_arc_tasks)}")


In [ ]:
# 1.3 — BARC (Berkeley synthetic ARC, ~200k via GPT-4)
from datasets import load_dataset
try:
    barc = load_dataset(
        "barc0/200k_HEAVY_gpt4o-description-gpt4omini-code_generated_problems",
        split="train",
    )
    print("BARC:", len(barc))
except Exception as e:
    print("BARC load skipped:", e); barc = None


In [ ]:
# 1.4 — Augmentation helpers (D4 + color permutation + example shuffle)
import copy

def rot90(g):  return [list(r) for r in zip(*g[::-1])]
def fliplr(g): return [r[::-1] for r in g]
D4 = [
    lambda g: g, rot90,
    lambda g: rot90(rot90(g)),
    lambda g: rot90(rot90(rot90(g))),
    fliplr,
    lambda g: rot90(fliplr(g)),
    lambda g: rot90(rot90(fliplr(g))),
    lambda g: rot90(rot90(rot90(fliplr(g)))),
]

def apply_d4(task, k):
    f = D4[k]
    return {"train":[{"input":f(e["input"]),"output":f(e["output"])} for e in task["train"]],
            "test": [{"input":f(e["input"]),"output":f(e["output"])} for e in task["test"]]}

def apply_color_perm(task, perm):
    def remap(g): return [[perm[c] for c in row] for row in g]
    return {"train":[{"input":remap(e["input"]),"output":remap(e["output"])} for e in task["train"]],
            "test": [{"input":remap(e["input"]),"output":remap(e["output"])} for e in task["test"]]}

def shuffle_examples(task, rng):
    t = copy.deepcopy(task); rng.shuffle(t["train"]); return t


In [ ]:
# 1.5 — Prompt format (matches 02_train_h100 + 04_eval_ttt for consistency)
SYSTEM = ("You are an expert ARC puzzle solver. Output only the test output grid as digits "
          "separated by spaces, one row per line. No explanations.")

def grid_to_str(g):
    return "\n".join(" ".join(str(c) for c in row) for row in g)

def parse_grid(s):
    rows = []
    for line in s.strip().splitlines():
        line = line.strip()
        if not line: continue
        try: rows.append([int(c) for c in line.split()])
        except: return None
    if not rows or any(len(r) != len(rows[0]) for r in rows): return None
    return rows

def format_user(task):
    parts = ["You are solving an ARC puzzle. Each grid is a 2D array of digits 0-9 (colors).",
             "Find the transformation rule from the input/output examples, then apply it to the test input.", ""]
    for i, ex in enumerate(task["train"], 1):
        parts.append(f"Example {i} Input:");  parts.append(grid_to_str(ex["input"]))
        parts.append(f"Example {i} Output:"); parts.append(grid_to_str(ex["output"]))
        parts.append("")
    parts.append("Test Input:");  parts.append(grid_to_str(task["test"][0]["input"]))
    parts.append(""); parts.append("Test Output:")
    return "\n".join(parts)

def format_assistant(task):
    return grid_to_str(task["test"][0]["output"])


In [ ]:
# 1.6 — Build the augmented base pool + tokenize/length-filter for SFT
def _coerce_examples(exs):
    """Each ex must be {'input': 2D-list, 'output': 2D-list}.
    BARC variants encode pairs as dict, list[2], or tuple[2] — handle all."""
    if not isinstance(exs, list): return None
    out = []
    for e in exs:
        if isinstance(e, dict) and "input" in e and "output" in e:
            inp, outp = e["input"], e["output"]
        elif isinstance(e, (list, tuple)) and len(e) == 2:
            inp, outp = e[0], e[1]
        else:
            return None
        if not (isinstance(inp, list) and inp and isinstance(inp[0], list)): return None
        if not (isinstance(outp, list) and outp and isinstance(outp[0], list)): return None
        out.append({"input": inp, "output": outp})
    return out

def normalize_barc(row):
    train, test = None, None
    if "train" in row and "test" in row:
        train = _coerce_examples(row["train"]); test = _coerce_examples(row["test"])
    elif "examples" in row and "test" in row:
        train = _coerce_examples(row["examples"]); test = _coerce_examples(row["test"])
    elif "examples" in row:
        ex = _coerce_examples(row["examples"])
        if ex and len(ex) >= 2: train, test = ex[:-1], [ex[-1]]
    return {"train": train, "test": test} if (train and test) else None

base_pool = list(arc_train) + list(re_arc_tasks)
if barc is not None:
    for r in barc.select(range(min(20000, len(barc)))):
        n = normalize_barc(r)
        if n and n["train"] and n["test"]:
            base_pool.append(n)
print(f"Base pool: {len(base_pool)}")

def gen_augmented(tasks, n_per_task, rng):
    out = []
    for t in tasks:
        for _ in range(n_per_task):
            x = apply_d4(t, rng.randrange(8))
            perm = list(range(10)); rng.shuffle(perm)
            x = apply_color_perm(x, perm)
            x = shuffle_examples(x, rng)
            out.append(x)
    return out

augmented = gen_augmented(base_pool, n_per_task=4, rng=random.Random(SEED))
print(f"Augmented samples (pre-filter): {len(augmented)}")


In [ ]:
# 1.7 — Load tokenizer + length-filter
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

# Qwen3 chat template supports a "thinking" mode; we want concise grids, no <think> blocks.
QWEN3_KW = {"enable_thinking": False} if "Qwen3" in MODEL_ID else {}

MAX_LEN = 3500

def to_chat_full(t):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":format_user(t)},
            {"role":"assistant","content":format_assistant(t)}]
    return tok.apply_chat_template(msgs, tokenize=False, **QWEN3_KW)

def to_chat_prompt(t):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":format_user(t)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **QWEN3_KW)

sft_texts = []
for t in augmented:
    s = to_chat_full(t)
    if len(tok(s, add_special_tokens=False).input_ids) <= MAX_LEN:
        sft_texts.append(s)
print(f"SFT samples after length filter: {len(sft_texts)}")

from datasets import Dataset
sft_dataset = Dataset.from_dict({"text": sft_texts}).shuffle(seed=SEED)
print(sft_dataset)


## 2 — Probe set

Three fixed ARC eval tasks we'll re-trace at each training stage. These are the "patients" — we image them at base, after SFT, after DPO, after GRPO, and watch how the network responds change.


In [ ]:
# 2.1 — Pick 3 small, diverse probes from the eval set
PROBE_IDXS = [0, 13, 47]  # arbitrary but fixed; small grids preferred
PROBES = [(arc_eval[i][0], arc_eval[i][1]) for i in PROBE_IDXS if i < len(arc_eval)]
for pid, t in PROBES:
    h = len(t["test"][0]["input"]); w = len(t["test"][0]["input"][0])
    print(f"  probe {pid}: {len(t['train'])} examples, test input {h}x{w}")


## 3 — Trace capture utilities

Three helpers that get called once at base + after each training stage:

1. **logit lens** — what does the LM head predict if we read it off layer L's hidden state, instead of the last layer? Reveals where the answer "crystallizes" in the network.
2. **layer norms + top neurons** — per-layer attention output norm, MLP output norm, and the top-10 most-activated MLP neurons. Forward hooks.
3. **LoRA snapshot** — full state-dict diff between phase boundaries.


In [ ]:
# 3.1 — Logit lens + per-token confidence/entropy
import numpy as np

@torch.inference_mode()
def capture_logit_lens(model, tok, probe_task, max_new=128, top_k_lens=1):
    """Generate up to max_new tokens; for each step, for each layer, run the LM head
    on the last-position hidden state and capture top-1 token + prob, plus chosen-token
    prob and entropy. Returns dict suitable for JSON."""
    base_for_head = model.get_base_model() if hasattr(model, "get_base_model") else model
    lm_head = base_for_head.get_output_embeddings()
    final_norm = getattr(base_for_head.model, "norm", None)  # Qwen2/3 uses RMSNorm at .model.norm

    prompt = to_chat_prompt(probe_task)
    inp = tok(prompt, return_tensors="pt").to(model.device)
    input_ids = inp.input_ids
    attn_mask = inp.attention_mask
    eos = tok.eos_token_id

    steps = []
    for step in range(max_new):
        out = model(input_ids=input_ids, attention_mask=attn_mask, output_hidden_states=True, use_cache=False)
        hs = out.hidden_states  # tuple of (n_layers+1) tensors [1, T, d]
        last_logits = out.logits[:, -1, :]
        probs_full  = torch.softmax(last_logits.float(), dim=-1)
        chosen_id   = int(torch.argmax(last_logits, dim=-1))
        chosen_prob = float(probs_full[0, chosen_id])
        ent         = float(-(probs_full * torch.log(probs_full.clamp_min(1e-12))).sum())

        layer_rows = []
        for li, h in enumerate(hs):
            h_last = h[:, -1, :].to(lm_head.weight.dtype)
            if final_norm is not None and li == len(hs) - 1:
                pass  # last hidden is post-norm in most arches
            elif final_norm is not None:
                h_last = final_norm(h_last)
            logits_l = lm_head(h_last).float()
            probs_l  = torch.softmax(logits_l, dim=-1)
            top1     = int(torch.argmax(probs_l, dim=-1))
            layer_rows.append({"layer": li, "top1_id": top1,
                               "top1_prob": float(probs_l[0, top1]),
                               "top1_tok": tok.decode([top1])})

        chosen_tok = tok.decode([chosen_id])
        steps.append({"step": step, "chosen_id": chosen_id, "chosen_tok": chosen_tok,
                      "chosen_prob": chosen_prob, "entropy": ent, "layers": layer_rows})

        # advance
        input_ids = torch.cat([input_ids, torch.tensor([[chosen_id]], device=model.device)], dim=1)
        attn_mask = torch.cat([attn_mask, torch.ones((1,1), device=model.device, dtype=attn_mask.dtype)], dim=1)
        if chosen_id == eos:
            break
    return {"steps": steps, "n_layers": len(steps[0]["layers"]) if steps else 0}


In [ ]:
# 3.2 — Forward-hook layer norms + top MLP neurons (single forward pass)
@torch.inference_mode()
def capture_layer_norms_and_neurons(model, tok, probe_task, top_k=10):
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    layers = base.model.layers
    attn_norms, mlp_norms, top_neurons = [], [], []

    handles = []
    cache = {}
    for i, layer in enumerate(layers):
        def make_hook_attn(idx):
            def h(_m, _i, o):
                t = o[0] if isinstance(o, tuple) else o
                cache[("attn", idx)] = t.detach().float().pow(2).mean(dim=-1).sqrt().mean().item()
            return h
        def make_hook_mlp(idx):
            def h(_m, _i, o):
                t = o
                cache[("mlp", idx)] = t.detach().float().pow(2).mean(dim=-1).sqrt().mean().item()
                # top-k neurons by mean |activation| across tokens
                act_abs = t.detach().float().abs().mean(dim=1).squeeze(0)  # [hidden]
                vals, idxs = torch.topk(act_abs, k=top_k)
                cache[("mlp_top", idx)] = (vals.tolist(), idxs.tolist())
            return h
        handles.append(layer.self_attn.register_forward_hook(make_hook_attn(i)))
        handles.append(layer.mlp.register_forward_hook(make_hook_mlp(i)))

    prompt = to_chat_prompt(probe_task) + format_assistant(probe_task)
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
    _ = model(**inp, use_cache=False)

    for h in handles: h.remove()

    n = len(layers)
    for i in range(n):
        attn_norms.append(cache.get(("attn", i), 0.0))
        mlp_norms.append(cache.get(("mlp",  i), 0.0))
        vals, idxs = cache.get(("mlp_top", i), ([], []))
        top_neurons.append({"layer": i, "vals": vals, "idxs": idxs})
    return {"attn_norms": attn_norms, "mlp_norms": mlp_norms, "top_neurons": top_neurons}


In [ ]:
# 3.3 — LoRA snapshot + delta
def snapshot_lora(model):
    snap = {}
    for n, p in model.named_parameters():
        if p.requires_grad and ("lora_A" in n or "lora_B" in n):
            snap[n] = p.detach().to("cpu", dtype=torch.float32).clone()
    return snap

def lora_delta(prev, curr):
    """Returns dict of {module_path: {prev_norm, curr_norm, delta_norm, pct}}.
    module_path strips lora_A/B suffix so we can aggregate per target module."""
    keys = set(prev.keys()) | set(curr.keys())
    by_mod = {}
    for k in keys:
        mod = k.replace(".lora_A.default.weight", "").replace(".lora_B.default.weight", "")
        mod = mod.replace(".lora_A.weight", "").replace(".lora_B.weight", "")
        a = prev.get(k); b = curr.get(k)
        a_n = float(a.norm()) if a is not None else 0.0
        b_n = float(b.norm()) if b is not None else 0.0
        d_n = float((b - a).norm()) if (a is not None and b is not None and a.shape == b.shape) else b_n
        e = by_mod.setdefault(mod, {"prev_norm": 0.0, "curr_norm": 0.0, "delta_norm": 0.0})
        e["prev_norm"]  += a_n
        e["curr_norm"]  += b_n
        e["delta_norm"] += d_n
    for m, v in by_mod.items():
        v["pct"] = (v["delta_norm"] / max(v["prev_norm"], 1e-9)) * 100.0
    return by_mod

def write_traces(stage, model, tok, probes):
    for pid, task in probes:
        ll = capture_logit_lens(model, tok, task, max_new=96)
        ln = capture_layer_norms_and_neurons(model, tok, task, top_k=10)
        out = {"probe_id": pid, "stage": stage, "logit_lens": ll, "layer_stats": ln}
        (RUN_DIR / "traces" / stage / f"probe_{pid}.json").write_text(json.dumps(out))
    print(f"[traces] wrote {len(probes)} probes for stage={stage}")


## 4 — Stage callbacks

Single `TrainerCallback` that streams `{step, loss, grad_norm, lr, ...}` to `runs/<RUN_ID>/logs/<phase>.jsonl`. Used for SFT, DPO, GRPO. Captures phase-specific extras (chosen/rejected/margin, rewards, KL) when present in the log dict.


In [ ]:
# 4.1 — TraceCallback
from transformers import TrainerCallback

class TraceCallback(TrainerCallback):
    def __init__(self, phase):
        self.phase = phase
        self.path = RUN_DIR / "logs" / f"{phase}.jsonl"
        self.f = open(self.path, "a")
    def on_log(self, args, state, control, logs=None, **kw):
        if logs is None: return
        rec = {"phase": self.phase, "step": state.global_step}
        for k in ("loss","grad_norm","learning_rate",
                 "rewards/chosen","rewards/rejected","rewards/margins",
                 "reward","reward_std","kl"):
            if k in logs: rec[k] = float(logs[k])
        self.f.write(json.dumps(rec) + "\n"); self.f.flush()
    def __del__(self):
        try: self.f.close()
        except: pass


## 5 — Stage 1: Base (no training, just trace)

Load base model in 4-bit + attach a fresh LoRA adapter (zero-initialized in lora_B). Capture probe traces. Pre-generate "rejected" completions for the DPO dataset while we're here.


In [ ]:
# 5.1 — Load model in 4-bit + LoRA
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
attn_impl = "flash_attention_2" if IS_H100 else "sdpa"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
    attn_implementation=attn_impl,
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=64, lora_alpha=128, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


In [ ]:
# 5.2 — Capture base traces + snapshot LoRA (= zeros) for delta math
write_traces("base", model, tok, PROBES)
SNAPSHOTS = {"base": snapshot_lora(model)}

# Save the (untrained) adapter so 'base' is reproducible
model.save_pretrained(str(RUN_DIR / "adapters" / "base"))


In [ ]:
# 5.3 — Pre-generate base-model rejected completions for DPO
# We sample on a small subset of the augmented pool. chosen = ground-truth grid string.
from tqdm.auto import tqdm

DPO_N = 1500
dpo_source = augmented[:DPO_N]

dpo_records = []
model.eval()
for t in tqdm(dpo_source, desc="DPO rejected gen"):
    user = format_user(t)
    chosen = format_assistant(t)
    msgs = [{"role":"system","content":SYSTEM},{"role":"user","content":user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **QWEN3_KW)
    inp = tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN-512).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inp, max_new_tokens=512, do_sample=True, temperature=1.0,
                             top_p=0.95, pad_token_id=tok.eos_token_id)
    rejected = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()
    if rejected and rejected != chosen:
        dpo_records.append({"prompt": text, "chosen": chosen, "rejected": rejected})

print(f"DPO records: {len(dpo_records)}")
(RUN_DIR / "logs" / "dpo_dataset_preview.json").write_text(json.dumps(dpo_records[:3], indent=2))


## 6 — Stage 2: SFT

TRL `SFTTrainer`, 1 epoch over augmented pool. Cosine LR + paged 8-bit AdamW + bf16 + gradient checkpointing. Capture traces after.


In [ ]:
# 6.1 — SFT
from trl import SFTTrainer, SFTConfig

sft_cfg = SFTConfig(
    output_dir=str(RUN_DIR / "adapters" / "sft"),
    num_train_epochs=1,
    per_device_train_batch_size=2 if VRAM_GB >= 70 else 1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=10,
    save_steps=10_000,
    save_total_limit=1,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_seq_length=MAX_LEN,
    dataset_text_field="text",
    packing=False,
    report_to="none",
    seed=SEED,
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tok,
    train_dataset=sft_dataset,
    args=sft_cfg,
    callbacks=[TraceCallback("sft")],
)
sft_trainer.train()
sft_trainer.model.save_pretrained(str(RUN_DIR / "adapters" / "sft"))
print("SFT done.")


In [ ]:
# 6.2 — Post-SFT traces + LoRA snapshot
write_traces("sft", model, tok, PROBES)
SNAPSHOTS["sft"] = snapshot_lora(model)


## 7 — Stage 3: DPO (preference optimization)

`chosen` = ground-truth grid string. `rejected` = base-model greedy/sampled completion (pre-generated in section 5.3). β=0.1, low LR. DPO needs a frozen reference; TRL handles this by snapshotting the model when `ref_model=None`.


In [ ]:
# 7.1 — DPO
from trl import DPOTrainer, DPOConfig

# Build HF dataset from records. DPO format: {prompt, chosen, rejected}.
# Note: trl >= 0.15 accepts a "prompt" string already chat-templated (full text up through assistant tag).
dpo_ds = Dataset.from_list(dpo_records)
print(dpo_ds)

dpo_cfg = DPOConfig(
    output_dir=str(RUN_DIR / "adapters" / "dpo"),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=10,
    save_steps=10_000,
    save_total_limit=1,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    beta=0.1,
    max_length=MAX_LEN,
    max_prompt_length=MAX_LEN-768,
    report_to="none",
    seed=SEED,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,           # PEFT model: TRL uses adapter-disabled forward as ref
    args=dpo_cfg,
    train_dataset=dpo_ds,
    tokenizer=tok,
    callbacks=[TraceCallback("dpo")],
)
dpo_trainer.train()
dpo_trainer.model.save_pretrained(str(RUN_DIR / "adapters" / "dpo"))
print("DPO done.")


In [ ]:
# 7.2 — Post-DPO traces + snapshot
write_traces("dpo", model, tok, PROBES)
SNAPSHOTS["dpo"] = snapshot_lora(model)


## 8 — Stage 4: GRPO (RL with verifiable grid-match reward)

Group-relative policy optimization. For each prompt we sample N completions, score them with a simple reward function, and update toward the high-reward ones relative to the group mean.

**Reward:** +1 exact grid match, +0.1 parses-as-grid, 0 otherwise.


In [ ]:
# 8.1 — GRPO dataset (prompts only, with ground truth carried for the reward fn)
GRPO_N = 600
grpo_source = augmented[DPO_N:DPO_N + GRPO_N]

grpo_rows = []
for t in grpo_source:
    grpo_rows.append({
        "prompt": to_chat_prompt(t),
        "ground_truth": format_assistant(t),
    })
grpo_ds = Dataset.from_list(grpo_rows)
print(grpo_ds)


In [ ]:
# 8.2 — Reward function
def grid_match_reward(completions, prompts=None, ground_truth=None, **kw):
    """Returns a list of floats, one per completion.
    TRL passes additional dataset columns by name as kwargs (e.g. ground_truth)."""
    rewards = []
    truths = ground_truth if ground_truth is not None else [None]*len(completions)
    for c, gt in zip(completions, truths):
        text = c if isinstance(c, str) else (c[0]["content"] if isinstance(c, list) else str(c))
        g = parse_grid(text)
        if g is None:
            rewards.append(0.0); continue
        if gt is not None and parse_grid(gt) == g:
            rewards.append(1.0)
        else:
            rewards.append(0.1)
    return rewards


In [ ]:
# 8.3 — GRPO training
from trl import GRPOTrainer, GRPOConfig

grpo_cfg = GRPOConfig(
    output_dir=str(RUN_DIR / "adapters" / "grpo"),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=5,
    save_steps=10_000,
    save_total_limit=1,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    num_generations=4,
    max_prompt_length=MAX_LEN-512,
    max_completion_length=512,
    beta=0.04,
    report_to="none",
    seed=SEED,
)

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=grid_match_reward,
    args=grpo_cfg,
    train_dataset=grpo_ds,
    processing_class=tok,
    callbacks=[TraceCallback("grpo")],
)
grpo_trainer.train()
grpo_trainer.model.save_pretrained(str(RUN_DIR / "adapters" / "grpo"))
print("GRPO done.")


In [ ]:
# 8.4 — Post-GRPO traces + snapshot
write_traces("grpo", model, tok, PROBES)
SNAPSHOTS["grpo"] = snapshot_lora(model)

# Persist snapshots to disk so we can recompute deltas later
import pickle
with open(RUN_DIR / "logs" / "lora_snapshots.pkl", "wb") as f:
    pickle.dump({k: {n: t.numpy() for n, t in v.items()} for k, v in SNAPSHOTS.items()}, f)


## 9 — Forensic interpretability (11 plotly views)

Every figure is also written to `runs/<RUN_ID>/viz/9_<id>.html` so you can open them later without rerunning.


In [ ]:
# 9.0 — Loader: reads every trace file into nested dicts.
import plotly.graph_objects as go
from plotly.subplots import make_subplots

STAGES = ["base", "sft", "dpo", "grpo"]

def load_all_traces():
    out = {s: {} for s in STAGES}
    for s in STAGES:
        for p in sorted((RUN_DIR / "traces" / s).glob("*.json")):
            d = json.loads(p.read_text())
            out[s][d["probe_id"]] = d
    return out

TRACES = load_all_traces()
PROBE_IDS = sorted({pid for s in STAGES for pid in TRACES[s].keys()})
print("Loaded probes:", PROBE_IDS, "stages:", STAGES)

VIZ_DIR = RUN_DIR / "viz"
def save_fig(fig, idx, title):
    fig.update_layout(title=title, template="plotly_white")
    path = VIZ_DIR / f"9_{idx}.html"
    fig.write_html(str(path), include_plotlyjs="cdn")
    print(f"[viz] {path}")


In [ ]:
# 9.1 — Logit Lens Heatmap Grid (probes x stages)
def viz_9_1(traces):
    if not PROBE_IDS: return None
    rows, cols = len(PROBE_IDS), 4
    fig = make_subplots(rows=rows, cols=cols,
                        row_titles=[f"probe {p}" for p in PROBE_IDS],
                        column_titles=STAGES,
                        horizontal_spacing=0.04, vertical_spacing=0.06)
    for r, pid in enumerate(PROBE_IDS, start=1):
        for c, s in enumerate(STAGES, start=1):
            d = traces[s].get(pid)
            if d is None: continue
            steps = d["logit_lens"]["steps"]
            if not steps: continue
            n_layers = len(steps[0]["layers"])
            Z = np.zeros((len(steps), n_layers))
            text = [["" for _ in range(n_layers)] for _ in range(len(steps))]
            for si, st in enumerate(steps):
                for li, lr in enumerate(st["layers"]):
                    Z[si, li] = lr["top1_prob"]
                    text[si][li] = f"L{li} step{si}: '{lr['top1_tok']}' p={lr['top1_prob']:.2f}"
            fig.add_trace(go.Heatmap(z=Z, text=text, hoverinfo="text",
                                     coloraxis="coloraxis", showscale=(r==1 and c==1)),
                          row=r, col=c)
    fig.update_layout(coloraxis={"colorscale":"Viridis","cmin":0,"cmax":1},
                      height=300*rows, width=1400)
    save_fig(fig, "1", "9.1 — Logit Lens Heatmap Grid (Y=gen step, X=layer, color=top-1 prob)")
    return fig

viz_9_1(TRACES)


In [ ]:
# 9.2 — Per-Token Confidence + Entropy
def viz_9_2(traces):
    if not PROBE_IDS: return None
    fig = make_subplots(rows=len(PROBE_IDS)*2, cols=1,
                        shared_xaxes=False, vertical_spacing=0.04,
                        row_titles=sum([[f"probe {p} P(chosen)", f"probe {p} entropy"] for p in PROBE_IDS], []))
    for ri, pid in enumerate(PROBE_IDS):
        for s in STAGES:
            d = traces[s].get(pid)
            if d is None: continue
            steps = d["logit_lens"]["steps"]
            xs = [st["step"] for st in steps]
            ps = [st["chosen_prob"] for st in steps]
            es = [st["entropy"] for st in steps]
            ts = [st["chosen_tok"] for st in steps]
            fig.add_trace(go.Bar(x=xs, y=ps, name=f"{s}", legendgroup=s,
                                 hovertext=ts, hoverinfo="text+y", showlegend=(ri==0)),
                          row=ri*2+1, col=1)
            fig.add_trace(go.Scatter(x=xs, y=es, mode="lines+markers", name=f"{s}",
                                     legendgroup=s, showlegend=False),
                          row=ri*2+2, col=1)
    fig.update_layout(height=300*len(PROBE_IDS)*2, width=1200, barmode="group")
    save_fig(fig, "2", "9.2 — Per-Token Confidence (top) + Entropy (bottom), per probe, all 4 stages")
    return fig

viz_9_2(TRACES)


In [ ]:
# 9.3 — Layer Activation Profiles (per probe, all stages overlaid)
def viz_9_3(traces):
    if not PROBE_IDS: return None
    fig = make_subplots(rows=len(PROBE_IDS), cols=2,
                        column_titles=["attn output norm", "mlp output norm"],
                        row_titles=[f"probe {p}" for p in PROBE_IDS])
    for ri, pid in enumerate(PROBE_IDS, start=1):
        for s in STAGES:
            d = traces[s].get(pid)
            if d is None: continue
            an = d["layer_stats"]["attn_norms"]
            mn = d["layer_stats"]["mlp_norms"]
            xs = list(range(len(an)))
            fig.add_trace(go.Scatter(x=xs, y=an, mode="lines+markers", name=s,
                                     legendgroup=s, showlegend=(ri==1)), row=ri, col=1)
            fig.add_trace(go.Scatter(x=xs, y=mn, mode="lines+markers", name=s,
                                     legendgroup=s, showlegend=False), row=ri, col=2)
    fig.update_xaxes(title_text="layer index")
    fig.update_layout(height=320*len(PROBE_IDS), width=1300)
    save_fig(fig, "3", "9.3 — Layer Activation Profiles (which layers do work?)")
    return fig

viz_9_3(TRACES)


In [ ]:
# 9.4 — Top-Neuron Heatmap (per probe x stage)
def viz_9_4(traces):
    if not PROBE_IDS: return None
    fig = make_subplots(rows=len(PROBE_IDS), cols=4,
                        column_titles=STAGES,
                        row_titles=[f"probe {p}" for p in PROBE_IDS],
                        horizontal_spacing=0.04, vertical_spacing=0.06)
    for ri, pid in enumerate(PROBE_IDS, start=1):
        for ci, s in enumerate(STAGES, start=1):
            d = traces[s].get(pid)
            if d is None: continue
            tn = d["layer_stats"]["top_neurons"]
            if not tn: continue
            Z = np.array([row["vals"] for row in tn])
            text = [[f"L{row['layer']} rank{j}: neuron #{row['idxs'][j]} val={row['vals'][j]:.3f}"
                     for j in range(len(row['vals']))] for row in tn]
            fig.add_trace(go.Heatmap(z=Z, text=text, hoverinfo="text",
                                     coloraxis="coloraxis", showscale=(ri==1 and ci==1)),
                          row=ri, col=ci)
    fig.update_layout(coloraxis={"colorscale":"Magma"},
                      height=320*len(PROBE_IDS), width=1400)
    save_fig(fig, "4", "9.4 — Top-10 MLP Neurons per Layer (Y=layer, X=top-k rank, color=|act|)")
    return fig

viz_9_4(TRACES)


In [ ]:
# 9.5 — LoRA Delta Bars (per phase boundary)
def viz_9_5():
    boundaries = [("sft","base"), ("dpo","sft"), ("grpo","dpo")]
    fig = make_subplots(rows=1, cols=len(boundaries),
                        column_titles=[f"{b}-{a}" for b,a in boundaries],
                        horizontal_spacing=0.18)
    for ci, (curr, prev) in enumerate(boundaries, start=1):
        if curr not in SNAPSHOTS or prev not in SNAPSHOTS: continue
        deltas = lora_delta(SNAPSHOTS[prev], SNAPSHOTS[curr])
        items = sorted(deltas.items(), key=lambda kv: -abs(kv[1]["delta_norm"]))[:20]
        names = [k.split(".")[-3] + "." + k.split(".")[-2] if "." in k else k for k, _ in items]
        vals  = [v["delta_norm"] for _, v in items]
        pcts  = [v["pct"] for _, v in items]
        text  = [f"{p:.1f}%" for p in pcts]
        fig.add_trace(go.Bar(x=vals, y=names, orientation="h",
                             text=text, textposition="outside",
                             marker_color=["#2a9d8f" if p>=0 else "#e76f51" for p in pcts]),
                      row=1, col=ci)
        fig.update_yaxes(autorange="reversed", row=1, col=ci)
    fig.update_layout(height=700, width=1500, showlegend=False)
    save_fig(fig, "5", "9.5 — Top-20 LoRA modules by |Δnorm| at each phase boundary")
    return fig

viz_9_5()


In [ ]:
# 9.6 — Gradient Flow Heatmap (per phase: step x summary stat)
# Note: we only logged a global grad_norm per step; we'll render that as a single-row
# heatmap per phase (the per-module breakdown would require deeper hooking).
def load_phase_log(phase):
    p = RUN_DIR / "logs" / f"{phase}.jsonl"
    if not p.exists(): return []
    return [json.loads(l) for l in p.read_text().splitlines() if l.strip()]

def viz_9_6():
    phases = ["sft","dpo","grpo"]
    fig = make_subplots(rows=len(phases), cols=1,
                        row_titles=phases, shared_xaxes=False, vertical_spacing=0.08)
    for ri, ph in enumerate(phases, start=1):
        rows = load_phase_log(ph)
        if not rows: continue
        steps = [r["step"] for r in rows if "grad_norm" in r]
        gns   = [r["grad_norm"] for r in rows if "grad_norm" in r]
        if not gns: continue
        Z = np.log10(np.array([gns]) + 1e-9)
        fig.add_trace(go.Heatmap(z=Z, x=steps, coloraxis="coloraxis"),
                      row=ri, col=1)
    fig.update_layout(coloraxis={"colorscale":"RdBu"}, height=240*len(phases), width=1300)
    save_fig(fig, "6", "9.6 — Gradient Flow Heatmap (log10 grad_norm vs step, per phase)")
    return fig

viz_9_6()


In [ ]:
# 9.7 — GRPO Reward Trajectory
def viz_9_7():
    rows = load_phase_log("grpo")
    if not rows:
        return None
    steps = [r["step"] for r in rows]
    rew   = [r.get("reward",   None) for r in rows]
    rstd  = [r.get("reward_std", None) for r in rows]
    kl    = [r.get("kl",       None) for r in rows]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["mean reward (line) + std (band)", "KL"])
    fig.add_trace(go.Scatter(x=steps, y=rew,  mode="lines+markers", name="reward"), row=1, col=1)
    fig.add_trace(go.Scatter(x=steps, y=rstd, mode="lines",          name="reward_std"), row=1, col=1)
    fig.add_trace(go.Scatter(x=steps, y=kl,   mode="lines",          name="kl"), row=2, col=1)
    fig.update_layout(height=600, width=1200)
    save_fig(fig, "7", "9.7 — GRPO Reward Trajectory")
    return fig

viz_9_7()


In [ ]:
# 9.8 — Cross-Stage Representation Drift (per probe, attn+mlp norms, all 4 stages)
def viz_9_8(traces):
    if not PROBE_IDS: return None
    fig = make_subplots(rows=len(PROBE_IDS), cols=1,
                        row_titles=[f"probe {p}" for p in PROBE_IDS])
    palette = {"base":"#264653","sft":"#2a9d8f","dpo":"#e9c46a","grpo":"#e76f51"}
    for ri, pid in enumerate(PROBE_IDS, start=1):
        for s in STAGES:
            d = traces[s].get(pid)
            if d is None: continue
            an = d["layer_stats"]["attn_norms"]
            mn = d["layer_stats"]["mlp_norms"]
            xs = list(range(len(an)))
            fig.add_trace(go.Scatter(x=xs, y=an, mode="lines",
                                     line=dict(color=palette[s], dash="solid"),
                                     name=f"{s} attn", legendgroup=s, showlegend=(ri==1)),
                          row=ri, col=1)
            fig.add_trace(go.Scatter(x=xs, y=mn, mode="lines",
                                     line=dict(color=palette[s], dash="dot"),
                                     name=f"{s} mlp", legendgroup=s, showlegend=(ri==1)),
                          row=ri, col=1)
    fig.update_xaxes(title_text="layer index")
    fig.update_layout(height=320*len(PROBE_IDS), width=1300)
    save_fig(fig, "8", "9.8 — Cross-Stage Representation Drift (solid=attn, dotted=mlp)")
    return fig

viz_9_8(TRACES)


In [ ]:
# 9.9 — Token-Level Cross-Stage Diff Table
def viz_9_9(traces):
    if not PROBE_IDS: return None
    figs = []
    for pid in PROBE_IDS:
        rows = []
        per = {s: traces[s].get(pid, {}).get("logit_lens", {}).get("steps", []) for s in STAGES}
        n = max((len(per[s]) for s in STAGES), default=0)
        for i in range(n):
            row = [i]
            toks = []
            for s in STAGES:
                if i < len(per[s]):
                    row.append(per[s][i]["chosen_tok"])
                    row.append(f"{per[s][i]['chosen_prob']:.2f}")
                    toks.append(per[s][i]["chosen_tok"])
                else:
                    row.append(""); row.append(""); toks.append(None)
            rows.append((row, toks))
        if not rows: continue
        # color cells red where token differs from base
        header = ["step"] + sum([[f"{s}.tok", f"{s}.prob"] for s in STAGES], [])
        cols = list(zip(*[r[0] for r in rows]))
        # build per-cell fill colors
        n_cols = len(header)
        fill = [["#ffffff"]*len(rows) for _ in range(n_cols)]
        for ri, (_, toks) in enumerate(rows):
            base_tok = toks[0]
            for si, s in enumerate(STAGES):
                if toks[si] is not None and base_tok is not None and toks[si] != base_tok:
                    fill[1 + si*2][ri] = "#fde2e2"
        fig = go.Figure(data=[go.Table(
            header=dict(values=header, fill_color="#264653", font=dict(color="white")),
            cells=dict(values=cols, fill_color=fill, align="left"),
        )])
        fig.update_layout(width=1300, height=min(900, 50 + 22*len(rows)))
        save_fig(fig, f"9_p{pid}", f"9.9 — Token-Level Diff (probe {pid}); pink = differs from base")
        figs.append(fig)
    return figs

viz_9_9(TRACES)


In [ ]:
# 9.10 — Master Training Curves (2x2)
def viz_9_10():
    sft  = load_phase_log("sft")
    dpo  = load_phase_log("dpo")
    grpo = load_phase_log("grpo")
    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=["loss + grad_norm (all phases)",
                                        "DPO chosen / rejected / margin",
                                        "GRPO reward",
                                        "GRPO KL"])
    for label, rows, color in [("sft", sft, "#2a9d8f"),
                                ("dpo", dpo, "#e9c46a"),
                                ("grpo", grpo, "#e76f51")]:
        st = [r["step"] for r in rows]
        lo = [r.get("loss") for r in rows]
        gn = [r.get("grad_norm") for r in rows]
        fig.add_trace(go.Scatter(x=st, y=lo, mode="lines", name=f"{label} loss",
                                 line=dict(color=color)), row=1, col=1)
        fig.add_trace(go.Scatter(x=st, y=gn, mode="lines", name=f"{label} gnorm",
                                 line=dict(color=color, dash="dot")), row=1, col=1)
    if dpo:
        st = [r["step"] for r in dpo]
        for k, name in [("rewards/chosen","chosen"),("rewards/rejected","rejected"),("rewards/margins","margin")]:
            ys = [r.get(k) for r in dpo]
            fig.add_trace(go.Scatter(x=st, y=ys, mode="lines", name=name), row=1, col=2)
    if grpo:
        st = [r["step"] for r in grpo]
        fig.add_trace(go.Scatter(x=st, y=[r.get("reward") for r in grpo], mode="lines",
                                 name="reward"), row=2, col=1)
        fig.add_trace(go.Scatter(x=st, y=[r.get("kl") for r in grpo], mode="lines",
                                 name="kl"), row=2, col=2)
    fig.update_layout(height=900, width=1500)
    save_fig(fig, "10", "9.10 — Master Training Curves")
    return fig

viz_9_10()


In [ ]:
# 9.11 — Final Summary Card
def viz_9_11(eval_scores):
    boundaries = [("sft","base"), ("dpo","sft"), ("grpo","dpo")]
    top_changes = []
    for curr, prev in boundaries:
        if curr in SNAPSHOTS and prev in SNAPSHOTS:
            d = lora_delta(SNAPSHOTS[prev], SNAPSHOTS[curr])
            top = sorted(d.items(), key=lambda kv: -abs(kv[1]["delta_norm"]))[:5]
            top_changes.append(f"{curr}-{prev}: " + ", ".join(k.split('.')[-2]+'.'+k.split('.')[-1] for k,_ in top))

    fields = [
        ("RUN_ID", RUN_ID),
        ("Model", MODEL_ID),
        ("GPU", f"{GPU_NAME} ({VRAM_GB:.0f} GB)"),
        ("Stages", " -> ".join(STAGES)),
        ("Optimizations", "QLoRA-NF4 / LoRA r=64 / FlashAttn-2 / bf16 / paged_adamw_8bit / grad-ckpt / D4+color-perm aug"),
        ("Probes", ", ".join(map(str, PROBE_IDS))),
    ]
    for s, sc in eval_scores.items():
        fields.append((f"acc[{s}]", f"{sc['correct']}/{sc['n']} = {sc['accuracy']:.1%}"))
    for line in top_changes:
        fields.append(("top LoRA Δ", line))
    fields.append(("artifacts", str(RUN_DIR)))

    fig = go.Figure(data=[go.Table(
        header=dict(values=["field","value"], fill_color="#264653", font=dict(color="white")),
        cells=dict(values=[[k for k,_ in fields],[v for _,v in fields]], align="left",
                   fill_color=[["#f4f4f4"]*len(fields), ["#ffffff"]*len(fields)]),
    )])
    fig.update_layout(width=1300, height=60 + 28*len(fields))
    save_fig(fig, "11", "9.11 — Final Summary Card")
    return fig


## 10 — Eval (greedy, first 50 ARC-AGI eval tasks at each stage)

For each stage, we *reload the matching adapter* on the same base model, run greedy decoding on the first 50 eval tasks, score exact-match. This produces the `acc[stage]` numbers on the summary card.


In [ ]:
# 10.1 — Eval helper
@torch.inference_mode()
def predict(m, task, max_new=2048):
    text = to_chat_prompt(task)
    inp = tok(text, return_tensors="pt").to(m.device)
    out = m.generate(**inp, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    return parse_grid(raw), raw

def eval_current(m, n=50):
    correct = 0; details = []
    for i, (name, task) in enumerate(arc_eval[:n]):
        pred, _ = predict(m, task)
        ok = (pred == task["test"][0]["output"])
        correct += int(ok)
        details.append({"task": name, "correct": ok})
    return {"n": n, "correct": correct, "accuracy": correct/n, "details": details}


In [ ]:
# 10.2 — Score every stage by reloading its adapter into the live PEFT model
from peft import PeftModel

eval_scores = {}
for s in STAGES:
    if s == "base":
        # disable adapters: equivalent to base predictions
        with model.disable_adapter():
            eval_scores[s] = eval_current(model, n=50)
    else:
        # reload that stage's saved adapter into the active model
        adapter_path = RUN_DIR / "adapters" / s
        # PEFT load_adapter swaps the active weights
        try:
            model.load_adapter(str(adapter_path), adapter_name="default", is_trainable=False)
        except Exception:
            # fallback: re-instantiate
            base_only = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, quantization_config=bnb, device_map={"":0},
                torch_dtype=torch.bfloat16, attn_implementation=attn_impl, trust_remote_code=True)
            model = PeftModel.from_pretrained(base_only, str(adapter_path))
        eval_scores[s] = eval_current(model, n=50)
    print(f"acc[{s}] = {eval_scores[s]['correct']}/{eval_scores[s]['n']} = {eval_scores[s]['accuracy']:.1%}")
    (RUN_DIR / "eval" / f"{s}.json").write_text(json.dumps(eval_scores[s], indent=2))

# Now render the summary card with eval numbers in it
viz_9_11(eval_scores)


## 11 — Bundle artifacts

Zips `runs/<RUN_ID>/` into one file. On Colab the second cell exposes a download button.


In [ ]:
# 11.1 — Zip the run directory
import shutil
zip_path = ROOT / f"{RUN_ID}.zip"
shutil.make_archive(str(zip_path).removesuffix(".zip"), "zip", RUN_DIR)
print("Bundle:", zip_path, f"({zip_path.stat().st_size/1e6:.1f} MB)")


In [ ]:
# 11.2 — Colab download (no-op outside Colab)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Outside Colab — pull the zip via your file browser:", zip_path)


## What you should expect

- **Accuracy curve:** monotone or near-monotone. Typical: base 5–10%, sft 25–40%, dpo +2–5pts, grpo +2–5pts.
- **9.1 Logit Lens:** after SFT you should see top-1 probability sharpen earlier in the network — answer crystallizes a few layers sooner.
- **9.5 LoRA Δ:** dominated by `q_proj` / `v_proj` after SFT; `o_proj` / `down_proj` light up more during GRPO as the model is pushed away from the reference.
- **9.7 Reward trajectory:** reward should climb and reward_std should *not* collapse to 0 (collapse = mode collapse, bad).
- **Total time on H100:** ~6–10 hours for the full pipeline at the defaults above. Cut `n_per_task` in 1.6 or `DPO_N` / `GRPO_N` to iterate faster.
